<a href="https://colab.research.google.com/github/Maestro-Titarenko/EconometricsProblemSets/blob/ECON771/PS7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 771 - Problem Set 7

In [17]:
# Install openpyxl if not have it
!pip install openpyxl

# Import necessary packages
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

## Exercise III

<blockquote>

<b>

This exercise is uses the Lending Club loan dataset. The outcome variable is a loan default status, *loan_status*, (=1 if paid off and =0 if not). This is going to be a DIY style exercise.

</b>

</blockquote>

**1. Fit a logistic regression model with 5-10 selected variables (converting the categorical variables to dummies). Give a short justification for why you excluded other variables. Report results in a table with standard errors and 95% confidence intervals (You are expected to code the log-likelihood function and optimize it numerically. The following guide may be helpful: https://indrag49.github.io/Numerical-Optimization/quasi-newton-methods.html).**

In [21]:
# Load data from a CSV file on GitHub
df = pd.read_csv('https://raw.githubusercontent.com/JiaxiLi1995/Econ771_python/main/ps7_data/lending_club_07_to_11_cleaned.csv')

# Load first few lines to have a feeling of the data
print(ps7_data.head())

# Step 1: Data Cleaning and Variable Selection

# 1. Inspect
print("Columns in dataset:")
print(loan_dataset.columns.tolist())
print(f"\nShape: {loan_dataset.shape}")
print(f"\nloan_status dtype: {loan_dataset['loan_status'].dtype}")
print(f"loan_status values:\n{loan_dataset['loan_status'].value_counts()}")

# 2. Outcome variable — loan_status is already 0/1 in this dataset
df['Y'] = df['loan_status'].astype(int)

print(f"Total loans: {len(df)}")
print(f"Paid (Y=1): {df['Y'].sum()}, Default (Y=0): {(1 - df['Y']).sum()}")
print(f"Default rate: {1 - df['Y'].mean():.3f}")

# 3. Clean int_rate (may be string with %)
print(f"\nint_rate dtype: {df['int_rate'].dtype}, sample: {df['int_rate'].head().tolist()}")
if df['int_rate'].dtype == object:
    df['int_rate'] = pd.to_numeric(
        df['int_rate'].astype(str).str.replace('%', '').str.strip(), errors='coerce'
    )

# 4. Clean term (may be string like " 36 months")
print(f"term dtype: {df['term'].dtype}, sample: {df['term'].head().tolist()}")
if df['term'].dtype == object:
    df['term'] = pd.to_numeric(
        df['term'].astype(str).str.replace('months', '').str.strip(), errors='coerce'
    )

df['term_60'] = (df['term'] == 60).astype(int)

# 5. Home ownership dummies (baseline = MORTGAGE)
print(f"\nhome_ownership values: {df['home_ownership'].value_counts().to_dict()}")
df['home_rent'] = (df['home_ownership'] == 'RENT').astype(int)
df['home_own'] = (df['home_ownership'] == 'OWN').astype(int)

# 6. Clean revol_util (may be string with %)
print(f"revol_util dtype: {df['revol_util'].dtype}, sample: {df['revol_util'].head().tolist()}")
if df['revol_util'].dtype == object:
    df['revol_util'] = pd.to_numeric(
        df['revol_util'].astype(str).str.replace('%', '').str.strip(), errors='coerce'
    )

# 7. Ensure numeric for all continuous variables
for col in ['loan_amnt', 'annual_inc', 'dti', 'revol_util', 'delinq_2yrs']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 8. Select covariates and drop missing
covariates = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
              'revol_util', 'delinq_2yrs', 'term_60', 'home_rent', 'home_own']

missing_cols = [c for c in covariates if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

df_clean = df[['Y'] + covariates].dropna()
print(f"\nRows before dropna: {len(df)}, after dropna: {len(df_clean)}")

# 9. Standardize continuous variables for numerical stability
continuous_vars = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'revol_util', 'delinq_2yrs']
means = df_clean[continuous_vars].mean()
stds = df_clean[continuous_vars].std()

df_std = df_clean.copy()
for var in continuous_vars:
    df_std[var] = (df_std[var] - means[var]) / stds[var]

print(f"\nClean sample size: {len(df_std)}")
print("\nSummary statistics (standardized):")
df_std[covariates].describe().round(3)

   Unnamed: 0  loan_amnt  funded_amnt  funded_amnt_inv       term int_rate  \
0           1       5000         5000           4975.0  36 months   10.65%   
1           2       2500         2500           2500.0  60 months   15.27%   
2           3       2400         2400           2400.0  36 months   15.96%   
3           4      10000        10000          10000.0  36 months   13.49%   
4           5       3000         3000           3000.0  60 months   12.69%   

   installment grade sub_grade                 emp_title  ... policy_code  \
0       162.87     B        B2                       NaN  ...           1   
1        59.83     C        C4                     Ryder  ...           1   
2        84.33     C        C5                       NaN  ...           1   
3       339.31     C        C1       AIR RESOURCES BOARD  ...           1   
4        67.79     B        B5  University Medical Group  ...           1   

  application_type  acc_now_delinq chargeoff_within_12_mths delinq_a

,loan_amnt,int_rate,annual_inc,dti,revol_util,delinq_2yrs,term_60,home_rent,home_own
count,39736.000,39736.000,39736.000,39736.000,39736.000,39736.000,39736.000,39736.000,39736.000
mean,0.000,0.000,-0.000,-0.000,-0.000,-0.000,0.269,0.475,0.077
std,1.000,1.000,1.000,1.000,1.000,1.000,0.443,0.499,0.267
min,-1.439,-1.772,-1.020,-1.995,-1.724,-0.298,0.000,0.000,0.000
25%,-0.769,-0.744,-0.446,-0.769,-0.828,-0.298,0.000,0.000,0.000
50%,-0.166,-0.044,-0.157,0.013,0.016,-0.298,0.000,0.000,0.000
75%,0.504,0.689,0.210,0.791,0.831,-0.298,1.000,1.000,0.000
max,3.183,3.372,93.023,2.497,1.801,22.078,1.000,1.000,1.000



**Selected covariates (9 variables including dummies):**

| Variable | Type | Rationale |
|----------|------|-----------|
| `loan_amnt` | Continuous | Larger loans carry more risk |
| `int_rate` | Continuous | Directly reflects assessed risk at origination |
| `annual_inc` | Continuous | Higher income aids repayment capacity |
| `dti` | Continuous | Debt-to-income ratio measures financial burden |
| `revol_util` | Continuous | Credit utilization proxies creditworthiness (substitutes for FICO, which is unavailable in this dataset) |
| `delinq_2yrs` | Continuous | Past delinquencies signal repayment behavior |
| `term_60` | Dummy | 60-month loans have longer exposure to default risk (baseline: 36 months) |
| `home_rent` | Dummy | Renting may indicate less financial stability (baseline: MORTGAGE) |
| `home_own` | Dummy | Owning may indicate more wealth (baseline: MORTGAGE) |

**Why exclude other variables?**

- **Post-origination variables** (`total_pymnt`, `last_pymnt_amnt`, `recoveries`, `collection_recovery_fee`, `out_prncp`): These are realized *after* the loan is issued. Using them would constitute data leakage — they are consequences, not predictors, of default.
- **High-cardinality categoricals** (`emp_title`, `title`, `desc`, `zip_code`, `addr_state`): Too many unique values; would require dimensionality reduction or regularization beyond the scope of this exercise.
- **Near-constant variables** (`policy_code`, `pymnt_plan`, `tax_liens`): Almost no variation in the sample, providing negligible predictive power.
- **Collinear with selected variables** (`funded_amnt` ≈ `loan_amnt`, `sub_grade` ≈ `int_rate`, `installment` ≈ f(`loan_amnt`, `int_rate`, `term`)): Would cause multicollinearity without adding independent information.
- **Joint application variables** (`application_type`): Nearly all loans are individual applications in this sample.

In [23]:

# Step 2: Logistic Log-Likelihood and BFGS Optimization


# 1. Build design matrix
Y = df_std['Y'].values                          # (n,)
X = df_std[covariates].values                    # (n, K)
X = np.column_stack([np.ones(len(X)), X])        # add intercept → (n, K+1)
n, K = X.shape
print(f"Sample size n = {n}, Number of parameters K = {K} (including intercept)\n")

# 2. Logistic CDF (numerically stable)
def logistic_cdf(z):
    return np.where(z >= 0,
                    1.0 / (1.0 + np.exp(-z)),
                    np.exp(z) / (1.0 + np.exp(z)))

# 3. Negative average log-likelihood
# Q_n(beta) = (1/n) sum_i [ Y_i log Lambda_i + (1 - Y_i) log(1 - Lambda_i) ]
# We minimize the negative of this.
def neg_log_likelihood(beta, Y, X):
    z = X @ beta
    Lambda = logistic_cdf(z)
    eps = 1e-15
    Lambda = np.clip(Lambda, eps, 1 - eps)
    ll = Y * np.log(Lambda) + (1 - Y) * np.log(1 - Lambda)
    return -np.mean(ll)

# 4. Gradient of the negative average log-likelihood
# grad Q_n = (1/n) sum_i (Y_i - Lambda_i) X_i
# So gradient of the negative is -(1/n) sum_i (Y_i - Lambda_i) X_i
def neg_log_likelihood_grad(beta, Y, X):
    z = X @ beta
    Lambda = logistic_cdf(z)
    residual = Y - Lambda
    return -X.T @ residual / len(Y)

# 5. Optimize via BFGS
print("Optimizing via BFGS...\n")
beta0 = np.zeros(K)

result = minimize(
    fun=neg_log_likelihood,
    x0=beta0,
    args=(Y, X),
    method='BFGS',
    jac=neg_log_likelihood_grad,
    options={'maxiter': 1000, 'disp': True}
)

beta_hat = result.x
print(f"\nConverged: {result.success}")
print(f"Iterations: {result.nit}")
print(f"Final neg log-likelihood: {result.fun:.6f}")

# 3: Sandwich Standard Errors and Results Table

# Under possible misspecification, the QMLE asymptotic variance is:
#   V = A^{-1} B A^{-1} / n
# where:
#   A = (1/n) sum_i Lambda_i (1 - Lambda_i) X_i X_i'   (negative Hessian)
#   B = (1/n) sum_i (Y_i - Lambda_i)^2 X_i X_i'        (outer product of scores)

z_hat = X @ beta_hat
Lambda_hat = logistic_cdf(z_hat)
residuals = Y - Lambda_hat

# A-hat
weights_A = Lambda_hat * (1 - Lambda_hat)
A_hat = (X.T * weights_A) @ X / n

# B-hat
weights_B = residuals ** 2
B_hat = (X.T * weights_B) @ X / n

# Sandwich variance
A_inv = np.linalg.inv(A_hat)
V_sandwich = A_inv @ B_hat @ A_inv / n
se_sandwich = np.sqrt(np.diag(V_sandwich))

# Classical MLE variance (for comparison): A^{-1} / n
V_classical = A_inv / n
se_classical = np.sqrt(np.diag(V_classical))

# 95% confidence intervals
z_crit = norm.ppf(0.975)
ci_lower = beta_hat - z_crit * se_sandwich
ci_upper = beta_hat + z_crit * se_sandwich

# --- Results Table ---
var_names = ['intercept'] + covariates

results_df = pd.DataFrame({
    'Variable': var_names,
    'Coefficient': beta_hat.round(4),
    'SE (Robust)': se_sandwich.round(4),
    'SE (Classical)': se_classical.round(4),
    'z-stat': (beta_hat / se_sandwich).round(3),
    'p-value': (2 * (1 - norm.cdf(np.abs(beta_hat / se_sandwich)))).round(4),
    'CI Lower (95%)': ci_lower.round(4),
    'CI Upper (95%)': ci_upper.round(4),
})

print("\n" + "=" * 95)
print("Logistic Regression Results (QMLE with Sandwich Standard Errors)")
print("=" * 95)
print(results_df.to_string(index=False))
print("-" * 95)
print(f"Observations: {n}")
print(f"Continuous variables are standardized (zero mean, unit variance).")
print(f"Home ownership baseline: MORTGAGE. Term baseline: 36 months.")
print(f"Robust SEs use the sandwich formula: V = A_inv @ B @ A_inv / n.")

# Display as formatted DataFrame
results_df

Sample size n = 39736, Number of parameters K = 10 (including intercept)

Optimizing via BFGS...

Optimization terminated successfully.
         Current function value: 0.384853
         Iterations: 46
         Function evaluations: 47
         Gradient evaluations: 47

Converged: True
Iterations: 46
Final neg log-likelihood: 0.384853

Logistic Regression Results (QMLE with Sandwich Standard Errors)
   Variable  Coefficient  SE (Robust)  SE (Classical)  z-stat  p-value  CI Lower (95%)  CI Upper (95%)
  intercept       2.0701       0.0273          0.0271  75.838   0.0000          2.0166          2.1236
  loan_amnt      -0.0051       0.0193          0.0175  -0.265   0.7914         -0.0430          0.0328
   int_rate      -0.4582       0.0201          0.0199 -22.839   0.0000         -0.4975         -0.4188
 annual_inc       0.3784       0.0467          0.0309   8.111   0.0000          0.2869          0.4698
        dti      -0.0147       0.0161          0.0158  -0.911   0.3623         -0.

,Variable,Coefficient,SE (Robust),SE (Classical),z-stat,p-value,CI Lower (95%),CI Upper (95%)
0,intercept,2.0701,0.0273,0.0271,75.838,0.0000,2.0166,2.1236
1,loan_amnt,-0.0051,0.0193,0.0175,-0.265,0.7914,-0.0430,0.0328
2,int_rate,-0.4582,0.0201,0.0199,-22.839,0.0000,-0.4975,-0.4188
3,annual_inc,0.3784,0.0467,0.0309,8.111,0.0000,0.2869,0.4698
4,dti,-0.0147,0.0161,0.0158,-0.911,0.3623,-0.0463,0.0169
5,revol_util,-0.0678,0.0184,0.0178,-3.696,0.0002,-0.1038,-0.0319
6,delinq_2yrs,0.0006,0.0142,0.0142,0.039,0.9686,-0.0273,0.0284
7,term_60,-0.4454,0.0376,0.0366,-11.853,0.0000,-0.5191,-0.3718
8,home_rent,-0.0091,0.0342,0.0328,-0.266,0.7901,-0.0760,0.0578
9,home_own,-0.0280,0.0590,0.0584,-0.475,0.6349,-0.1437,0.0876


<b>
    
2. To evaluate the predictive performance of the model, split the data randomly into two subsets of equal size (training and test set). Fit the logistic regression on the training set and compute predicted default status for the tests set. The predictions can be obtained as
$$\hat{y}_i = 1 \quad if \ and \ only \ if \quad x^\top_i \hat{\beta} \geq 0.$$

What is the fraction of the incorrectly predicted $\hat{y}_i$ in the test set?

</b>

In [26]:

# 4: Train/Test Split and Predictive Performance


# --- Split data randomly into two equal subsets ---
np.random.seed(42)
indices = np.random.permutation(n)
n_train = n // 2

train_idx = indices[:n_train]
test_idx = indices[n_train:]

Y_train, X_train = Y[train_idx], X[train_idx]
Y_test, X_test = Y[test_idx], X[test_idx]

print(f"Training set size: {len(train_idx)}")
print(f"Test set size:     {len(test_idx)}")

# --- Re-fit logistic regression on training set only ---
print("\nFitting on training set via BFGS...")

result_train = minimize(
    fun=neg_log_likelihood,
    x0=np.zeros(K),
    args=(Y_train, X_train),
    method='BFGS',
    jac=neg_log_likelihood_grad,
    options={'maxiter': 1000, 'disp': True}
)

beta_hat_train = result_train.x
print(f"\nConverged: {result_train.success}")

# --- Predict on test set ---
# Rule from the problem set: y_hat_i = 1 iff x_i^T beta_hat >= 0
linear_pred = X_test @ beta_hat_train
Y_hat = (linear_pred >= 0).astype(int)

# --- Misclassification rate ---
incorrect = np.sum(Y_hat != Y_test)
misclass_rate = incorrect / len(Y_test)

print(f"\n{'='*55}")
print(f"Predictive Performance on Test Set")
print(f"{'='*55}")
print(f"Total test observations:    {len(Y_test)}")
print(f"Incorrectly predicted:      {incorrect}")
print(f"Misclassification rate:     {misclass_rate:.4f} ({misclass_rate*100:.2f}%)")
print(f"Accuracy:                   {1 - misclass_rate:.4f} ({(1-misclass_rate)*100:.2f}%)")

# --- Confusion matrix ---
TP = np.sum((Y_hat == 1) & (Y_test == 1))
TN = np.sum((Y_hat == 0) & (Y_test == 0))
FP = np.sum((Y_hat == 1) & (Y_test == 0))
FN = np.sum((Y_hat == 0) & (Y_test == 1))

print(f"\nConfusion Matrix:")
print(f"{'':>25} Predicted=1   Predicted=0")
print(f"  Actual=1 (Paid)       {TP:>8}      {FN:>8}")
print(f"  Actual=0 (Default)    {FP:>8}      {TN:>8}")
print(f"\nPrecision: {TP/(TP+FP):.4f}   (of those predicted paid, how many actually paid)")
print(f"Recall:    {TP/(TP+FN):.4f}   (of those actually paid, how many we caught)")

Training set size: 19868
Test set size:     19868

Fitting on training set via BFGS...
Optimization terminated successfully.
         Current function value: 0.381873
         Iterations: 44
         Function evaluations: 45
         Gradient evaluations: 45

Converged: True

Predictive Performance on Test Set
Total test observations:    19868
Incorrectly predicted:      2868
Misclassification rate:     0.1444 (14.44%)
Accuracy:                   0.8556 (85.56%)

Confusion Matrix:
                          Predicted=1   Predicted=0
  Actual=1 (Paid)          17000             1
  Actual=0 (Default)        2867             0

Precision: 0.8557   (of those predicted paid, how many actually paid)
Recall:    0.9999   (of those actually paid, how many we caught)
